In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

from machine_intelligence.utilities.data_handling import load_parquet_table



In [ ]:
from machine_intelligence.utilities.viz_setup import figure_setup, VariableNames

cwd = Path.cwd()
cwd

In [ ]:

project_root = Path(cwd).parents[1]
project_root

In [ ]:

current_date = datetime.now().strftime("%Y-%m-%d")
plot_output_dir = project_root / "plot_outputs" / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)

table_outputs = project_root / "table_outputs" / current_date
table_outputs.mkdir(parents=True, exist_ok=True)


In [ ]:
cohort_stats_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/data_quality/2024-08-02/dataset/sepsis_type_a_cohort_time_series/sepsis_type_a_cohort_time_series_descriptive_stats.csv")

In [ ]:
df_cohort_stats = pd.read_csv(cohort_stats_path).set_index("Unnamed: 0")

df_cohort_stats

In [ ]:
df_cohort_stats_summary = df_cohort_stats.loc[:, ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
df_cohort_stats_summary = VariableNames.get_human_readable_index(df_cohort_stats_summary)

df_cohort_stats_summary = df_cohort_stats_summary.drop(
    index=["Patient Unit Stay ID", "Time Offset", "Sepsis Present", "Sepsis Patient"])
cols_to_convert = df_cohort_stats_summary.columns.difference(["count"])
df_cohort_stats_summary[cols_to_convert] = df_cohort_stats_summary[cols_to_convert].apply(pd.to_numeric)

df_cohort_stats_summary

In [ ]:

df_cohort_stats_summary["count"] = df_cohort_stats_summary["count"].astype(int)
df_cohort_stats_summary = df_cohort_stats_summary.round(decimals=2)

In [ ]:
df_cohort_stats_summary.to_csv(path_or_buf=table_outputs / "cohort_stats_summary.csv", sep=';', decimal=".",
                               float_format='%.2f', )


In [ ]:
cohort_stats_static_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/data_quality/2024-08-02/dataset/sepsis_type_a_cohort_static/sepsis_type_a_cohort_static_descriptive_stats.csv")


In [ ]:
df_cohort_static_stats = pd.read_csv(cohort_stats_static_path).set_index("Unnamed: 0")
df_cohort_static_stats

In [ ]:

cohort_stats_static_categorical_path = project_root / Path(
    "eicu-code-dhm/intermediary_data/data_quality/2024-08-02/dataset/sepsis_type_a_cohort_static/sepsis_type_a_cohort_static_categorical_data.csv")

df_cohort_static_cat_stats = pd.read_csv(cohort_stats_static_categorical_path)
df_cohort_static_cat_stats


summary stats + manually calculated

In [ ]:
gender_male = df_cohort_static_cat_stats.loc[0, ["count"]]
gender_female = df_cohort_static_cat_stats.loc[1, ["count"]]

age_stats = pd.to_numeric(df_cohort_static_cat_stats["age"]).describe()

unitadmitsource_ed = df_cohort_static_cat_stats.loc[0, ["count.9"]]
unitadmitsource_da = df_cohort_static_cat_stats.loc[1, ["count.9"]]

mort_alive = df_cohort_static_cat_stats.loc[0, ["count.7"]]
mort_expired = df_cohort_static_cat_stats.loc[1, ["count.7"]]

feature_cat_keep = ["actualiculos", "actualventdays", "thrombolytics", "aids", "hepaticfailure", "lymphoma",
                    "metastaticcancer",
                    "leukemia",
                    "immunosuppression", "cirrhosis", "electivesurgery", "ima", "diabetes",
                    "unitvisitnumber", "unitdischargeoffset", "admissionweight", "admissionheight"]

In [ ]:
df_cohort_static_cat_stats.head()

In [ ]:

df_cohort_static_cat_stats_summary = df_cohort_static_stats.loc[feature_cat_keep, ["count", "mean", "std", "min",
                                                                                   "25%", "50%", "75%", "max"]]
df_cohort_static_cat_stats_summary

In [ ]:

# df_cohort_static_cat_stats_summary = pd.concat([df_cohort_static_cat_stats_summary, pd.DataFrame(age_stats).T])
# df_cohort_static_cat_stats_summary

In [ ]:


df_cohort_static_cat_stats_summary

In [ ]:

df_cohort_static_cat_stats_summary["count"] = df_cohort_static_cat_stats_summary["count"].astype(int)
df_cohort_static_cat_stats_summary = df_cohort_static_cat_stats_summary.round(decimals=2)
df_cohort_static_cat_stats_summary.to_csv(path_or_buf=table_outputs / "cohort_stats_statics_summary.csv", sep=';', decimal=".",
                                          float_format='%.2f', )



In [ ]:
static_path = project_root / Path("eicu-code-dhm/intermediary_data/dataset/sepsis-type-a-cohort/2024-08-02/dataset_static.parquet")


In [ ]:
df_static = load_parquet_table(static_path, backend="numpy_nullable")

In [ ]:
df_static["age"].dropna().astype("int").describe()
# note i'll take the description from pycharm its more precise

In [ ]:
df_static.describe()

In [ ]:
df_static["unittype"].value_counts()

In [ ]:
df_static["aids"].value_counts(normalize=True)

In [ ]:

df_static["lymphoma"].value_counts(normalize=True)


In [ ]:
df_static["metastaticcancer"].value_counts(normalize=True)


In [ ]:

df_static["leukemia"].value_counts(normalize=True)


In [ ]:

df_static["immunosuppression"].value_counts(normalize=True)


In [ ]:

df_static["cirrhosis"].value_counts(normalize=True)


In [ ]:
df_static["diabetes"].value_counts(normalize=True)


The dataset is at this point only cohorted, so represents the cohorts data without any alterations further down the line for the actual experiment

In [ ]:
ts_path = project_root / Path("eicu-code-dhm/intermediary_data/dataset/sepsis-type-a-cohort/2024-08-02/dataset_ts.parquet")

df_ts = load_parquet_table(ts_path, backend="numpy_nullable")


In [ ]:
df_ts.describe()

In [ ]:
df_ts["patientunitstayid"].drop_duplicates().describe()

So 51766 patients in the time series table

In [ ]:
df_cohort_static_stats["aids"]